# P2 D-256 StyleGAN2 — Colab Training (Drive-based)

**Workflow:** `MyDrive/osai/` 에 repo 통째로 두고 작업. 코드와 ckpt는 Drive에 그대로, **데이터 zip만 Colab 로컬로 1회 cp** (Drive over FUSE는 random-access I/O가 매우 느려 학습 throughput에 치명적).

**Hardware:** A100 GPU (Runtime → Change runtime type → A100).

**Pre-flight:**
- `MyDrive/osai/` 가 이미 있어야 함. 없으면 셀 3이 자동 clone.
- `MyDrive/osai/p2/train_50k_256.zip` 위치에 학습 데이터 두기.
- Colab Secrets (왼쪽 사이드바 🔑)에 `WANDB_API_KEY` 등록 + 노트북 접근 권한 토글 ON.

**Mount stale 대응:** 셀 2가 helper `ensure_drive()`를 정의해두고, launch/resume 셀이 실행 직전에 호출해 끊긴 mount를 자동 재연결합니다.

In [ ]:
# 1. GPU check — must be A100 (or L4 if A100 unavailable)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# 2. Mount Drive + define ensure_drive() for auto-remount on stale FUSE
import os, time
from google.colab import drive

def _drive_is_alive() -> bool:
    try:
        os.listdir('/content/drive/MyDrive')
        return True
    except OSError:
        return False

def ensure_drive(force: bool = False) -> None:
    """Mount /content/drive if not mounted, or remount if the FUSE is stale."""
    if force or not _drive_is_alive():
        drive.mount('/content/drive', force_remount=True)
        # brief settle
        for _ in range(10):
            if _drive_is_alive():
                break
            time.sleep(0.5)
    print('Drive alive:', _drive_is_alive())

ensure_drive()

In [ ]:
# 3. Sync repo into MyDrive/osai (clone if absent, pull if present)
ensure_drive()
import os
WORK = '/content/drive/MyDrive/osai'
if not os.path.exists(WORK):
    !git clone https://github.com/geniemo/osai.git {WORK}
elif not os.path.exists(f'{WORK}/.git'):
    print(f'{WORK} exists but is not a git repo. Either delete it or clone elsewhere.')
else:
    %cd {WORK}
    !git fetch origin
    !git checkout improve
    !git pull --rebase origin improve
%cd {WORK}
!git rev-parse --short HEAD

In [ ]:
# 4. Install dependencies
!pip install -q pyyaml wandb pytorch-fid onnx onnxruntime scipy

In [ ]:
# 5. Copy training zip from Drive → Colab local disk (CRITICAL for throughput)
# Drive FUSE has terrible random-access perf; ZipImageDataset reads a random
# entry per sample, so we MUST have the zip on Colab-local /content/ for training.
ensure_drive()
import os, shutil
src = '/content/drive/MyDrive/osai/p2/train_50k_256.zip'
dst_dir = '/content/p2_data'
dst = f'{dst_dir}/train_50k_256.zip'
os.makedirs(dst_dir, exist_ok=True)
if not os.path.exists(dst):
    print(f'Copying {src} → {dst}…')
    shutil.copy(src, dst)
print('Local zip:', os.path.getsize(dst) / 1e9, 'GB')

# Symlink to the config's expected path inside Drive repo
link = '/content/drive/MyDrive/osai/p2/data/train_50k_256.zip'
os.makedirs(os.path.dirname(link), exist_ok=True)
if os.path.islink(link) or os.path.exists(link):
    os.remove(link)
os.symlink(dst, link)
!ls -la /content/drive/MyDrive/osai/p2/data/

In [ ]:
# 6. WandB auth — Colab Secret
# Pre-req: Colab 왼쪽 사이드바 🔑 'Secrets'에서 이름 `WANDB_API_KEY`로 등록 +
# 'Notebook access' 토글 ON.
import os
try:
    from google.colab import userdata
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    print('WANDB_API_KEY loaded from Colab Secret')
except Exception as e:
    print(f'Secret lookup failed ({e}); falling back to interactive login.')
    import wandb
    wandb.login()

## Launch — first session

Working dir is `MyDrive/osai/`. ckpts will land in `MyDrive/osai/p2/runs/d256_main/` (Drive) so they survive Colab disconnects automatically — no separate backup needed.

If you saw `OSError: [Errno 107] Transport endpoint is not connected`, the launch cell now auto-remounts the Drive before the cd.

In [ ]:
# Re-verify mount + cd via os.chdir (more robust to stale handles than %cd)
ensure_drive()
import os
os.chdir('/content/drive/MyDrive/osai')
!pwd && mkdir -p p2/runs/d256_main
!PYTHONPATH=. WANDB_API_KEY=$WANDB_API_KEY python p2/train.py --config p2/configs/d256.yaml 2>&1 | tee -a p2/runs/d256_main/train.log

## Resume — after disconnect

Re-run cells 2 (mount), 3 (git pull), 4 (pip), 5 (data copy — Colab local was wiped), 6 (WandB secret). Then this cell auto-finds the latest ckpt in Drive:

In [ ]:
ensure_drive()
import os, glob
os.chdir('/content/drive/MyDrive/osai')
ckpts = sorted(glob.glob('p2/runs/d256_main/ckpt_*.pt'))
latest = ckpts[-1] if ckpts else None
print('Latest ckpt:', latest)
if latest:
    !PYTHONPATH=. WANDB_API_KEY=$WANDB_API_KEY python p2/train.py --config p2/configs/d256.yaml --resume {latest} 2>&1 | tee -a p2/runs/d256_main/train.log

## Self-measure FID (between sessions)

Optional but recommended. Requires `MyDrive/osai/p2/valid_10k_256.zip`. Real-stats cached once.

In [ ]:
# One-time: copy valid zip locally, extract, cache real-stats
ensure_drive()
import os, shutil
src = '/content/drive/MyDrive/osai/p2/valid_10k_256.zip'
dst = '/content/p2_data/valid_10k_256.zip'
if not os.path.exists(dst):
    shutil.copy(src, dst)
valid_dir = '/content/p2_data/valid_10k_256_dir'
if not os.path.isdir(valid_dir):
    os.makedirs(valid_dir, exist_ok=True)
    !cd {valid_dir} && unzip -q -o {dst}
os.chdir('/content/drive/MyDrive/osai')
!python -m pytorch_fid {valid_dir} --save-stats p2/checkpoints/fid_stats_256.npz

In [ ]:
# Measure FID on latest ckpt (or specify path)
ensure_drive()
import os, glob
os.chdir('/content/drive/MyDrive/osai')
ckpts = sorted(glob.glob('p2/runs/d256_main/ckpt_*.pt'))
latest = ckpts[-1] if ckpts else None
print('Evaluating:', latest)
if latest:
    !PYTHONPATH=. python p2/eval_fid.py --ckpt {latest} --stats p2/checkpoints/fid_stats_256.npz --n 8000 --batch 32

## ONNX export — for leaderboard

In [ ]:
ensure_drive()
import os
os.chdir('/content/drive/MyDrive/osai')
!PYTHONPATH=. python p2/export_onnx.py \
    --ckpt p2/runs/d256_main/final.pt \
    --out p2/checkpoints/model.onnx
# model.onnx is now in MyDrive/osai/p2/checkpoints/ — download from Drive UI